In [1]:
import pickle
import sys
import os
import networkx as nx

# Add parent directory to path for imports
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))
from utils.graph_utils import (build_knn_graph_from_features, save_graph, set_random_seeds, load_graph, 
                              get_device, compute_auxiliary_features, compute_topological_features,
                              prepare_pytorch_geometric_data)

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Test FireGNN on Their Image data

In [3]:
G = load_graph("../datasets/G_Organcmnist_inductive.gpickle")
print(nx.is_connected(G))
print('Number of connected components:', nx.number_connected_components(G))

Loaded graph from ../datasets/G_Organcmnist_inductive.gpickle: 23583 nodes, 82315 edges
False
Number of connected components: 121


In [4]:
topo_features = compute_topological_features(G)
topo_features

Computing topological features...


Computing two-hop agreement: 100%|██████████| 23583/23583 [00:01<00:00, 17133.05it/s]


{'degrees': array([ 7,  7,  7, ...,  6,  4, 11]),
 'clustering': array([0.0952381 , 0.14285714, 0.14285714, ..., 0.        , 0.33333333,
        0.01818182]),
 'two_hop_agreement': array([0.96551724, 0.64705882, 0.95      , ..., 0.89285714, 0.63157895,
        0.83333333]),
 'eigenvector_centrality': array([1.31661456e-05, 3.69877065e-08, 2.53912504e-04, ...,
        1.66790147e-06, 4.72049064e-07, 7.12302197e-05]),
 'degree_centrality': array([0.00029684, 0.00029684, 0.00029684, ..., 0.00025443, 0.00016962,
        0.00046646]),
 'avg_edge_weight': array([0.92591325, 0.92618476, 0.91980682, ..., 0.9239049 , 0.93452545,
        0.91430251])}

In [5]:
data = prepare_pytorch_geometric_data(G, topo_features)
data

Data(x=[23583, 512], edge_index=[2, 164630], edge_attr=[164630], y=[23583], train_mask=[23583], val_mask=[23583], test_mask=[23583], topo_features=[23583, 6], topo_scaler_mean=[6], topo_scaler_scale=[6])

In [6]:
data.topo_scaler_mean

tensor([6.9809e+00, 1.7921e-01, 7.2609e-01, 1.2804e-03, 2.9603e-04, 9.2411e-01])

Extract features, labels from Their Graph and rebuld the graph with different ks

In [9]:
features = []
labels = []
train_mask = []
val_mask = []
test_mask = []
for node_id, attrs in G.nodes(data=True):
    x = attrs['x']
    features.append(x)
    y = attrs['y']
    labels.append(y)
    train = attrs['train']
    train_mask.append(train)
    val = attrs['val']
    val_mask.append(val)
    test = attrs['test']
    test_mask.append(test)

In [12]:
import numpy as np


features = np.asarray(features)
labels = np.asarray(labels)
labels.shape

(23583,)